# mmb0-1b_smb1-1b_2p Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 06.04.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

## 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import logging
import os
import warnings
import pickle
import random
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
try:
    import squidpy as sq
except:
    import dask
    dask.config.set({"dataframe.query-planning": True})
    import dask.dataframe.dask_expr
    print("Could not import squidpy...")
    import rapids_singlecell as rsc
from datasets import load_from_disk
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from app.infer import infer
from app.train import train
from nichejepa.datasets_new import prepare_dataset
from nichejepa.utils_new.config import create_params_from_YAML_wandb_config
from nichejepa.utils_new.distributed import init_distributed
from nichejepa.utils_new.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [ ]:
# Set params
dataset = 'mmb0-1b_smb1-1b_1p'
tokenizer = 'cell_graph'
# load_timestamp = '29032025_080927_875' # gt_tiny
# load_timestamp = '03042025_224852_256' # gt_small
# load_timestamp = '27032025_163938_061' # gt_base
# load_timestamp = '03042025_224013_784' # gt_large
# load_timestamp = None
load_timestamp = '24122025_144827_523'
model_type = 'test'
read_checkpoint = 'nichejepa-ep99.pth.tar' # 'nichejepa-ep1_6.pth.tar'

In [ ]:
emb_layer = None
n_layer = 3
top_k = None
batch_size = 128
pin_memory = False
num_workers = 12
n_pcs = 0

In [ ]:
model_config_folder_path = '/nfs/team361/sb75/nichejepa/configs/model'
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
datasets_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS'
token_dict_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_mus_musculus.pkl'

In [ ]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [ ]:
# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'{model_config_folder_path}/{dataset}_{model_type}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
if load_timestamp:
    params['state']['load_checkpoint'] = True

params

In [ ]:
# Define folder path for saving and loading artifacts
if not load_timestamp:
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifacts_folder_path,
                               params['data']['dataset_name'],
                               current_timestamp)
else:
    folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'
    
folder_path

## 1.3 Run Notebook Setup

In [ ]:
os.makedirs(f'{folder_path}/figures', exist_ok=True)

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [ ]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      agg_type='avg',
                      masked_tokens=None,
                      agg_excluded_tokens=None,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[])

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    for obs_col in obs_cols:
        adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
def compute_embedding_similarity(adata,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [ ]:
def compute_neighborhood_umap(adata,
                              write_to_disk=True):
    if 'neighborhood_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) neighborhood gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'neighborhood_emb_layer_{layer}',
                        key_added='neighborhood')
    if 'neighborhood_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='neighborhood',
                   key_added='neighborhood_umap')
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    if 'cell_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) cell gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'cell_emb_layer_{layer}',
                        key_added='cell')
    if 'cell_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='cell',
                   key_added='cell_umap')

        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell':
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood':
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            sc.tl.leiden(adata,
                         neighbors_key=emb_key,
                         key_added=latent_cluster_key,
                         resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        metrics_df = pd.DataFrame({
            'metric': ['NMI', 'ARI'],
            'value': [nmi, ari]
        })
        metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                          index=False)
        
        # Plot UMAPs
        os.makedirs(f'{folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(custom_palette if label == latent_cluster_key else None),
                           show=False)
                plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                            dpi=300,
                            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')
            os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
                      f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata

# 2. Analysis

In [ ]:
# Load data
if params['data']['model_input_size'] == 704:
    dataset = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/mmb0-1b_smb1-1b_1p/mmb0-1b_smb1-1b_1p_32_None_None_None_None_None_shifted_log_knn_10_704.dataset')
elif params['data']['model_input_size'] == 2816:
    dataset = load_from_disk(f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/mmb0-1b_smb1-1b_1p/mmb0-1b_smb1-1b_1p_32_None_None_None_None_None_shifted_log_knn_10_2816.dataset')

In [ ]:
# Minimal torch formatting per split
def set_minimal_format(d):
    cols = [c for c in d.column_names if c != 'cell_id' or not False]
    d.set_format(type="torch", columns=cols, output_all_columns=False)
    return d

dataset = set_minimal_format(dataset)

In [ ]:
devices=['cuda:0']
world_size=1
os.environ['CUDA_VISIBLE_DEVICES'] = str(devices[rank].split(':')[-1])

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

world_size, rank = init_distributed(rank_and_world_size=(rank, world_size), port=random.randint(40000, 50000))
logger.info(f'Running... (rank: {rank}/{world_size})')

## 2.1 Run Inference

### 2.1.1 xhs49-1b_1p (Xenium Human Skin Melanoma)

#### 2.1.1.1 Define Config

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
mask_unshared_tokens = False
exclude_unshared_tokens = False
feature_norm = False
params['data']['batch_size'] = 32
params['data']['test_batch_ids'] = [
    '0_batch82',
    '1_batch15']

In [ ]:
# Set dataset params
dataset_name = 'mmb0-1b_smb1-1b_1p'
batch_key = 'batch'
emb_key = 'neighborhood'
latent_leiden_resolution = 0.2
spot_sizes = [0.3, 30]

latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

#### 2.1.1.2 Run Model Inference

In [ ]:
if mask_unshared_tokens or exclude_unshared_tokens:
    mmb0_adata_raw = sc.read_h5ad('/home/aih/sebastian.birk/workspace/projects/nichejepa-reproducibility/data/st_data/silver/mmb0-1b_smb1-1b_2p/starmap_plus_mouse_cns_batch15.h5ad')
    smb1_adata_raw = sc.read_h5ad('/home/aih/sebastian.birk/workspace/projects/nichejepa-reproducibility/data/st_data/silver/mmb0-1b_smb1-1b_2p/merfish_mouse_brain_batch82.h5ad')
    
    token_dict_path = '/home/aih/sebastian.birk/workspace/projects/nichejepa-reproducibility/data/tokenizer_data/token_dictionary_mus_musculus.pkl'
    
    with open(token_dict_path, 'rb') as f:
        token_dict = pickle.load(f) 
    
    mmb0_gene_tokens = [token_dict[gene] for gene in mmb0_adata_raw.var['ensembl_id'].tolist()]
    smb1_gene_tokens = [token_dict[gene] for gene in smb1_adata_raw.var['ensembl_id'].tolist()]
    
    overlapping_gene_tokens = set(mmb0_gene_tokens) & set(smb1_gene_tokens)
    non_overlapping_gene_tokens = (set(mmb0_gene_tokens) | set(smb1_gene_tokens)) - overlapping_gene_tokens
    overlapping_gene_tokens = list(overlapping_gene_tokens)
    non_overlapping_gene_tokens = list(non_overlapping_gene_tokens)
else:
    overlapping_gene_tokens = None
    non_overlapping_gene_tokens = None

In [ ]:
obs_cols = [
    'donor_id',
    'brain_section_label',
    'major_brain_region',
    'ccf_region_name',
    'Main_molecular_cell_type',
    'Sub_molecular_cell_type',
    'Main_molecular_tissue_region',
    'Sub_molecular_tissue_region',
    'Molecular_spatial_cell_type',
    'dataset',
    'cell_type_ontology_term_id',
    'nichecompass_latent_cluster']

In [ ]:
params

In [ ]:
agg_type = 'avg'

In [ ]:
dataset['batch_value'][-1] + torch.tensor([21957])

In [ ]:
# Run model inference
adata = infer(params,
              dataset,
              obs_cols=obs_cols,
              load_folder_path=folder_path,
              agg_type=agg_type,
              masked_tokens=non_overlapping_gene_tokens if mask_unshared_tokens else None,
              agg_excluded_tokens=non_overlapping_gene_tokens if exclude_unshared_tokens else None,
              #cell_gene_ids=[1031],
              #neighborhood_gene_ids=[1031]
             )

# Save adata
adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_mask_unshared_tokens" if mask_unshared_tokens else ""}{"_exclude_unshared_tokens" if exclude_unshared_tokens else ""}{"_feature_norm" if feature_norm else ""}.h5ad')

In [ ]:
adata

In [ ]:
niche_key = 'nichecompass_latent_cluster'

compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

In [ ]:
adata

In [ ]:
Z = adata.obsm['neighborhood_emb_layer_3']

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())


In [ ]:
std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

In [ ]:
import numpy as np

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)


In [ ]:
niche_key = 'niche_type'

In [ ]:
adata

In [ ]:
# Extract batch from cell ID
adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]

In [ ]:
n_neighbors = 15
emb_key = 'neighborhood_emb'
emb_layer = f'{emb_key}_layer_3'
latent_leiden_resolution = 0.6
latent_cluster_key = f"{emb_key}_clusters_res_{latent_leiden_resolution}"

In [ ]:
rsc.pp.neighbors(
    adata,
    n_neighbors=n_neighbors,
    use_rep=emb_layer,
    n_pcs=None,
    key_added=emb_key)

rsc.tl.umap(adata,
            neighbors_key=emb_key,
            key_added=f'{emb_key}_umap')

In [ ]:
rsc.tl.leiden(adata,
              neighbors_key=emb_key,
              key_added=latent_cluster_key,
              resolution=latent_leiden_resolution)

In [ ]:
obs_col = 'nichecompass_latent_cluster'
adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']

In [ ]:
spot_sizes = [100, 20, 20]

In [ ]:
import sys
sys.path.append("../../utils")
from analysis_utils import *

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot UMAPs
#os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', obs_col, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    if len(np.unique(adata.obs[label])) > 1:
        sc.pl.umap(adata,
                   neighbors_key=emb_key,
                   title=(emb_key + "_emb_" + label if emb_key not in label else label),
                   color=label,
                   palette=(custom_palette if label == latent_cluster_key else None),
                   show=False)
        plt.show()
        #plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
        #            dpi=300,
        #            bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  #save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'
                 )
    #os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
    #          f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))

#if write_to_disk:
    # Save adata
    #adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')


In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
labels = adata.obs[obs_col]
labels_pred = adata.obs[latent_cluster_key]
nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
ari = adjusted_rand_score(labels, labels_pred)
print(f'NMI is {nmi}. ARI is {ari}.')
metrics_df = pd.DataFrame({
    'metric': ['NMI', 'ARI'],
    'value': [nmi, ari]
})

In [ ]:
import numpy as np

X = adata.obsm['neighborhood_emb_layer_3']  # shape (N, D)

# Normalize rows
X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)

# Compute and normalize mean vector
mean_vec = X.mean(axis=0, keepdims=True)
mean_norm = mean_vec / np.linalg.norm(mean_vec)

# Row-wise cosine similarity
cos_sim = (X_norm * mean_norm).sum(axis=1)

# Average
mean_cos = cos_sim.mean()

print(mean_cos)

In [ ]:
n_neighbors = 15
emb_key = 'cell_emb'
emb_layer = f'{emb_key}_layer_3'
latent_leiden_resolution = 0.6
latent_cluster_key = f"{emb_key}_clusters_res_{latent_leiden_resolution}"

In [ ]:
rsc.pp.neighbors(
    adata,
    n_neighbors=n_neighbors,
    use_rep=emb_layer,
    n_pcs=None,
    key_added=emb_key)

rsc.tl.umap(adata,
            neighbors_key=emb_key,
            key_added=f'{emb_key}_umap')

In [ ]:
rsc.tl.leiden(adata,
              neighbors_key=emb_key,
              key_added=latent_cluster_key,
              resolution=latent_leiden_resolution)

In [ ]:
obs_col = 'nichecompass_latent_cluster'
adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']

In [ ]:
spot_sizes = [100, 20, 20]

In [ ]:
import sys
sys.path.append("../../utils")
from analysis_utils import *

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot UMAPs
#os.makedirs(f'{folder_path}/figures', exist_ok=True)
labels = ['batch', obs_col, latent_cluster_key]
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    if len(np.unique(adata.obs[label])) > 1:
        sc.pl.umap(adata,
                   neighbors_key=emb_key,
                   title=(emb_key + "_emb_" + label if emb_key not in label else label),
                   color=label,
                   palette=(custom_palette if label == latent_cluster_key else None),
                   show=False)
        plt.show()
        #plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
        #            dpi=300,
        #            bbox_inches='tight')

# Plot niches in tissue
batch_ids = adata.obs['batch'].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs['batch'] == batch_id]
    sc.pl.spatial(adata_batch,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[i],
                  #save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'
                 )
    #os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
    #          f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))

#if write_to_disk:
    # Save adata
    #adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())


In [ ]:
import numpy as np

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)
